# S3.22 - Custom logging in Databricks
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** A log is a record of what happened that survives the run: levels, timestamps, structure

In [0]:
# ============================================================
# S3.22 — Cell 2
# Custom Logging:
# Python logging, levels, exception logging, timing,
# structured JSON logs, and Spark-based log analysis
#
# Goal:
# 1. Create a Python logger with timestamps and log levels.
# 2. Understand DEBUG / INFO / WARNING / ERROR / CRITICAL.
# 3. See how changing the logger threshold changes output.
# 4. Log before and after a Spark action.
# 5. Measure Spark-action duration correctly.
# 6. Log exceptions with useful context and traceback.
# 7. Create structured JSON logs using Python logging.
# 8. Query structured logs using a Spark DataFrame.
# 9. Understand notebook/driver logging vs durable
#    production logging.
#
# Environment:
# Databricks Serverless / Spark Connect
#
# Important:
# - No spark.sparkContext or RDD APIs are used.
# - Logs are stored only in Python memory for this lesson.
# - This is NOT durable production logging.
# ============================================================

import logging
import json
import time
from datetime import datetime, timezone


print("=== S3.22: CUSTOM LOGGING ===")
print(f"Spark version: {spark.version}")
print()

In [0]:
# ============================================================
# Step 1 — Custom In-Memory Handler
#
# Python logging normally sends records to handlers such as:
#
# console
# file
# external logging platform
#
# For this controlled lesson we use a custom handler that
# stores formatted log records in a Python list.
#
# This allows us to inspect the logs without needing storage.
# ============================================================

class ListHandler(logging.Handler):

    def __init__(self):
        super().__init__()

        self.records = []

    def emit(self, record):

        formatted_record = (
            self.format(record)
        )

        self.records.append(
            formatted_record
        )

# ------------------------------------------------------------
# Create the logger.
# ------------------------------------------------------------

logger = logging.getLogger("retailpulse.s3_22")

# Remove handlers left over from a previous notebook run.
logger.handlers.clear()

# Prevent messages from also being sent to the root logger.
logger.propagate = False

# ------------------------------------------------------------
# Create our in-memory handler.
# ------------------------------------------------------------

handler = ListHandler()

# Handler itself does not filter by severity.
# Filtering in this exercise is controlled by logger.setLevel().

handler.setLevel(logging.NOTSET)

# ------------------------------------------------------------
# Human-readable formatter.
#
# Logging Formatter normally uses local time.
# Setting converter=time.gmtime makes %(asctime)s UTC.
# ------------------------------------------------------------

human_formatter = logging.Formatter(
    "%(asctime)s UTC | "
    "%(levelname)-8s | "
    "%(name)s | "
    "%(message)s"
)

human_formatter.converter = (time.gmtime)

handler.setFormatter(human_formatter)

logger.addHandler(handler)


print("Logger created:")
print(logger.name)
print()

In [0]:
# ============================================================
# Step 2 — Logging Levels
#
# Standard severity order:
#
# DEBUG
# INFO
# WARNING
# ERROR
# CRITICAL
#
# Lower numeric severity:
#
# DEBUG
#
# Higher severity:
#
# CRITICAL
#
# Logger threshold means:
#
# keep events at that severity OR ABOVE it.
# ============================================================

print("=== STEP 2A: LOGGER LEVEL = INFO ===")

handler.records.clear()

logger.setLevel(logging.INFO)

logger.debug("DEBUG message — should NOT be kept")

logger.info("INFO message — should be kept")

logger.warning("WARNING message — should be kept")

logger.error("ERROR message — should be kept")

logger.critical("CRITICAL message — should be kept")

print(f"Records kept: "f"{len(handler.records)}")

for record in handler.records:

    print(record)

print()

# ------------------------------------------------------------
# Change the threshold to WARNING.
#
# DEBUG and INFO are now below the threshold.
# ------------------------------------------------------------

print("=== STEP 2B: LOGGER LEVEL = WARNING ===")

handler.records.clear()

logger.setLevel(logging.WARNING)

logger.debug("DEBUG message — should NOT be kept")

logger.info("INFO message — should NOT be kept")

logger.warning("WARNING message — should be kept")

logger.error("ERROR message — should be kept")

logger.critical("CRITICAL message — should be kept")

print(f"Records kept: "f"{len(handler.records)}")

for record in handler.records:

    print(record)

print()

# ------------------------------------------------------------
# Restore INFO for the remaining exercises.
# ------------------------------------------------------------

logger.setLevel(
    logging.INFO
)


In [0]:
# ============================================================
# Step 3 — Log Around a Spark Action
#
# Reading the table builds a DataFrame.
#
# df.count()
#
# is the Spark action that triggers execution.
#
# We log:
#
# start
#      ↓
# Spark action
#      ↓
# done + rows + duration
#
# time.perf_counter()
#
# is used for elapsed-time measurement.
# ============================================================

print("=== STEP 3: LOG AROUND A SPARK ACTION ===")

handler.records.clear()

df_orders = (spark.table("samples.tpch.orders"))

logger.info("step=count_orders status=start")

start_time = (time.perf_counter())

order_count = (df_orders.count())

elapsed_seconds = (

    time.perf_counter() - start_time)

logger.info(
    "step=count_orders "
    "status=done "
    "rows=%s "
    "seconds=%.2f",
    order_count,
    elapsed_seconds
)

for record in handler.records:

    print(record)

print()

In [0]:
# ============================================================
# Step 4 — Exception Logging
#
# Reuse the malformed cast concept from S3.19.
#
# Under ANSI mode:
#
# "abc"
#    ↓
# cast as INT
#    ↓
# can raise a conversion exception
#
#
# logger.exception(...)
#
# should be called inside except.
#
# It logs:
#
# ERROR-level message
# +
# exception context
# +
# traceback
#
#
# We record the ACTUAL exception type rather than
# hard-coding one.
# ============================================================

print("=== STEP 4: LOG AN EXCEPTION ===")


handler.records.clear()


ansi_enabled = (
    spark.conf.get(
        "spark.sql.ansi.enabled"))

print(f"ANSI mode: {ansi_enabled}")


bad_df = (
    spark.createDataFrame(
        [
            ("1",),
            ("abc",)
        ],
        ["x"]
    )
)


cast_status = None
cast_error_type = None
cast_error_message = None

try:
    (
        bad_df.selectExpr(
        "cast(x as int) as n").collect())
    
    cast_status = "done"
    logger.info("step=cast_x status=done")

except Exception as e:

    cast_status = "failed"

    cast_error_type = (type(e).__name__)

    cast_error_message = (str(e)[:500])

    logger.exception(
        "step=cast_x "
        "status=failed "
        "error_type=%s",
        cast_error_type)

for record in handler.records:

    print(record)

In [0]:
# ============================================================
# Step 5 — Why logger.exception Is Useful
#
# logger.error(...)
#
# normally logs only the message unless exception information
# is explicitly supplied.
#
#
# logger.exception(...)
#
# used inside except automatically includes exception
# information / traceback.
# ============================================================

print("=== STEP 5: EXCEPTION RESULT ===")

print(f"Cast status     : "f"{cast_status}")

print(f"Exception type  : "
    f"{cast_error_type}")

print(f"Message preview : "f"{cast_error_message}")

print()

In [0]:
# ============================================================
# Step 6 — Structured JSON Logging
#
# Human-readable logs are useful when a person reads them.
#
# Structured logs are more useful when logs must be:
#
# searched
# filtered
# aggregated
# queried
# monitored
#
# Instead of:
#
# "count finished with 1500 rows"
#
# structured logging records:
#
# {
#   "step": "count_orders",
#   "status": "done",
#   "rows": 1500
# }
#
#
# We now create a JSON formatter for Python logging.
# ============================================================

class JsonFormatter(logging.Formatter):

    def format(self, record):

        event = {

            "ts": (datetime.now(timezone.utc).isoformat()),

            "level": (record.levelname),

            "logger": (record.name),

            "message": (record.getMessage()),

            "step": (getattr(record,"step",None)),

            "status": (getattr(record,"status",None)),

            "rows": (getattr(record,"rows",None)),

            "seconds": (getattr(record,"seconds",None)),

            "error_type": (getattr(record,"error_type",None))}

        # If logger.exception() supplied exception information,
        # preserve it as structured text.
        if record.exc_info:

            event["exception"] = (self.formatException(record.exc_info))

        else:
            event["exception"] = None

        return json.dumps(
            event,
            default=str)

In [0]:
# ============================================================
# Step 7 — Structured Logger
#
# Use a separate logger so human-readable and structured
# examples remain easy to understand independently.
# ============================================================

json_logger = logging.getLogger("retailpulse.s3_22.structured")

json_logger.handlers.clear()

json_logger.propagate = False

json_logger.setLevel(logging.INFO)

json_handler = ListHandler()

json_handler.setLevel(logging.NOTSET)

json_handler.setFormatter(JsonFormatter())

json_logger.addHandler(json_handler)

print("=== STEP 7: STRUCTURED LOGGER CREATED ===")

print(json_logger.name)

print()

In [0]:
# ============================================================
# Step 8 — Write Structured Events
#
# Use logging's extra={} argument to attach structured fields.
#
# These values come from ACTUAL execution above.
# ============================================================

json_handler.records.clear()

# ------------------------------------------------------------
# Event 1 — Successful Spark count
# ------------------------------------------------------------

json_logger.info(

    "orders counted",

    extra={

        "step": "count_orders",

        "status": "done",

        "rows": order_count,
        
        "seconds": round(elapsed_seconds,2)})

# ------------------------------------------------------------
# Event 2 — Cast result
#
# Do not hard-code failed/succeeded.
# Use what actually happened.
# ------------------------------------------------------------

if cast_status == "failed":

    json_logger.error("cast operation failed",

        extra={

            "step": "cast_x",

            "status": cast_status,

            "error_type": cast_error_type})

else:
    json_logger.info("cast operation completed",

        extra={

            "step": "cast_x",

            "status": cast_status})
# ------------------------------------------------------------
# Event 3 — Synthetic learning event
#
# This is intentionally labelled synthetic.
# It is NOT claiming that we actually wrote Silver data.
# ------------------------------------------------------------

json_logger.info("synthetic learning event",

    extra={

        "step": "example_write_silver",

        "status": "demo",

        "rows": 1500})


print("=== STEP 8: STRUCTURED JSON RECORDS ===")

for record in json_handler.records:

    print(record)

print()

In [0]:
# ============================================================
# Step 9 — Convert Structured Logs to Python Dictionaries
#
# ListHandler stored JSON strings.
#
# Convert:
#
# JSON string
#     ↓
# Python dictionary
# ============================================================

structured_events = [

    json.loads(record)

    for record in json_handler.records
]

print("=== STEP 9: STRUCTURED EVENT DICTIONARIES ===")


for event in structured_events:

    print(event)

print()

In [0]:
# ============================================================
# Step 10 — Structured Logs as a Spark DataFrame
#
# IMPORTANT:
#
# We use an explicit schema instead of schema inference.
#
# Why?
#
# Some structured-log fields can be NULL in every event.
#
# Example:
#
# exception = None
#
# If Spark only sees None values for a field, it cannot infer
# whether that column should be STRING, INT, DOUBLE, etc.
#
# That can cause:
#
# [CANNOT_DETERMINE_TYPE]
#
# For structured logs, an explicit schema is also better
# because the log contract remains stable.
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)


log_schema = StructType([

    StructField(
        "ts",
        StringType(),
        True
    ),

    StructField(
        "level",
        StringType(),
        True
    ),

    StructField(
        "logger",
        StringType(),
        True
    ),

    StructField(
        "message",
        StringType(),
        True
    ),

    StructField(
        "step",
        StringType(),
        True
    ),

    StructField(
        "status",
        StringType(),
        True
    ),

    StructField(
        "rows",
        LongType(),
        True
    ),

    StructField(
        "seconds",
        DoubleType(),
        True
    ),

    StructField(
        "error_type",
        StringType(),
        True
    ),

    StructField(
        "exception",
        StringType(),
        True
    )
])


df_logs = (
    spark.createDataFrame(
        structured_events,
        schema=log_schema
    )
)


print("=== STEP 10: STRUCTURED LOG SCHEMA ===")


df_logs.printSchema()


print()


print("=== STRUCTURED LOG DATA ===")


df_logs.show(truncate=False)

In [0]:
# ============================================================
# Step 11 — Query Structured Logs
#
# Once logs become structured data, we can use normal
# DataFrame operations.
# ============================================================

print(
    "=== STEP 11: QUERY LOGS WITH SPARK ==="
)


(
    df_logs

    .select(
        "ts",
        "level",
        "step",
        "status",
        "rows",
        "seconds",
        "error_type"
    )

    .show(
        truncate=False
    )
)


# ------------------------------------------------------------
# Failed steps
# ------------------------------------------------------------

failed_logs = (

    df_logs

    .filter(
        "status = 'failed'"
    )
)


failed_count = (
    failed_logs.count()
)


print(
    f"Failed log events: "
    f"{failed_count}"
)


failed_logs.select(
    "step",
    "status",
    "error_type"
).show(
    truncate=False
)


In [0]:
# ============================================================
# Step 12 — Aggregate Logs
#
# Structured logs can also be grouped and summarized.
# ============================================================

print(
    "=== STEP 12: LOG EVENTS BY STATUS ==="
)


(
    df_logs

    .groupBy(
        "status"
    )

    .count()

    .orderBy(
        "status"
    )

    .show()
)

In [0]:
# ============================================================
# Step 13 — Logger Threshold vs Handler Threshold
#
# Both loggers and handlers can filter records.
#
# In this exercise:
#
# handler level = NOTSET
#
# Therefore:
#
# logger.setLevel(...)
#
# controls the threshold.
#
#
# In a larger application you might have:
#
# Logger = DEBUG
#
# Console handler = INFO
#
# Error handler = ERROR
#
# allowing different destinations to receive different
# severities.
# ============================================================

print(
    "=== STEP 13: LOGGER vs HANDLER LEVEL ==="
)


print(
    f"Human logger level : "
    f"{logging.getLevelName(logger.level)}"
)

print(
    f"Human handler level: "
    f"{logging.getLevelName(handler.level)}"
)

print(
    f"JSON logger level  : "
    f"{logging.getLevelName(json_logger.level)}"
)

print(
    f"JSON handler level : "
    f"{logging.getLevelName(json_handler.level)}"
)

print()


In [0]:
# ============================================================
# Step 14 — Driver / Notebook Logging Boundary
#
# The Python logger used here executes in the notebook's
# Python process.
#
# It should NOT be interpreted as automatically collecting
# every Spark executor/task log.
#
#
# Conceptually:
#
# Notebook Python process
#         ↓
# Python logger
#
#
# Spark distributed execution
#         ↓
# executor/task/runtime logging
#
#
# They are related operationally but are not the same thing.
# ============================================================

print(
    "=== STEP 14: LOGGING BOUNDARY ==="
)


print("""
This lesson demonstrates Python-side notebook logging.

It does NOT mean this ListHandler automatically captures
all Spark executor/task/runtime logs.

The custom ListHandler exists only in the Python process
running this notebook.
""")

In [0]:
# ============================================================
# Step 15 — In-Memory vs Production Logging
# ============================================================

print(
    "=== STEP 15: IN-MEMORY vs PRODUCTION LOGGING ==="
)


print("""
THIS LAB

ListHandler
    ↓
Python list
    ↓
easy to inspect
    ↓
lost when the process/session ends


PRODUCTION

Application / pipeline
    ↓
structured logs
    ↓
durable centralized destination
    ↓
search
    ↓
monitoring
    ↓
alerts
    ↓
retention
    ↓
access control


Therefore:

ListHandler
→ excellent for learning/testing

ListHandler
→ NOT a durable production log store
""")

In [0]:
# ============================================================
# Step 16 — Operational Logging Guidance
#
# Good logs should answer:
#
# WHAT happened?
# WHERE did it happen?
# WHEN did it happen?
# WAS it successful?
# HOW MUCH was processed?
# HOW LONG did it take?
# WHAT failed?
#
# Useful fields often include:
#
# run_id
# pipeline
# notebook
# step
# status
# rows
# duration
# source
# target
# error_type
#
#
# Never log:
#
# passwords
# tokens
# API keys
# sensitive payloads
# unnecessary personal data
# ============================================================

print("=== STEP 16: GOOD LOGGING PRACTICE ===")

print("""
Useful operational fields:

run_id
pipeline
notebook
step
status
rows
duration
source
target
error_type


Avoid logging secrets such as:

passwords
access tokens
API keys
connection secrets


Logging should improve observability
without creating a security problem.
""")

In [0]:
# ============================================================
# Step 17 — Final Learning Summary
# ============================================================

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
LOG LEVELS

DEBUG
→ detailed diagnostic information

INFO
→ normal operational progress

WARNING
→ unexpected condition, but processing can continue

ERROR
→ an operation failed

CRITICAL
→ severe failure that may threaten the application/process


LOGGER THRESHOLD

INFO threshold:

DEBUG
→ filtered

INFO and above
→ kept


WARNING threshold:

DEBUG / INFO
→ filtered

WARNING / ERROR / CRITICAL
→ kept


SPARK ACTION LOGGING

log start
    ↓
start timer
    ↓
Spark action
    ↓
stop timer
    ↓
log rows + duration


EXCEPTION LOGGING

try
    ↓
Spark action
    ↓
except
    ↓
logger.exception(...)
    ↓
context + exception + traceback


STRUCTURED LOGGING

event
    ↓
JSON fields
    ↓
Spark DataFrame
    ↓
filter / group / analyze


IMPORTANT

Human-readable logs
→ optimized for people

Structured JSON logs
→ optimized for machines and analytics

SERVERLESS / SPARK CONNECT

Use:
spark.createDataFrame(...)

Do NOT rely on:
spark.sparkContext.parallelize(...)

BOUNDARY

Python notebook logging
≠
automatic Spark executor logging


PRODUCTION PRINCIPLE

Useful logs should be:

structured
contextual
searchable
durable
secure
monitorable
""")

print("=== S3.22 CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# S3.22 - Cell 3: compact checkpoint (prints RESULT lines only)
#
# Goal:
# 1. Records kept at INFO vs WARNING
# 2. rows and seconds logged around a Spark action
# 3. An exception logged with context
# 4. Structured JSON logs queried with Spark
# ============================================================
import logging, json, time
from datetime import datetime, timezone

# ------------------------------------------------------------
# Step 1 - In-memory logger
# ------------------------------------------------------------
class ListHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))

log = logging.getLogger("retailpulse.s3_22_check")
log.handlers.clear()
log.propagate = False
h = ListHandler()
h.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
log.addHandler(h)

# ------------------------------------------------------------
# Step 2 - Level thresholds
# ------------------------------------------------------------
for level in (logging.INFO, logging.WARNING):
    h.records.clear()
    log.setLevel(level)
    log.debug("d"); log.info("i"); log.warning("w"); log.error("e")
    print(f"RESULT 1 threshold {logging.getLevelName(level):<7}: kept {len(h.records)} -> {h.records}")

# ------------------------------------------------------------
# Step 3 - Log around a Spark action
# ------------------------------------------------------------
log.setLevel(logging.INFO)
h.records.clear()
t0 = time.time()
n = spark.table("samples.tpch.orders").count()
log.info(f"step=read_orders status=done rows={n} seconds={time.time()-t0:.2f}")
print("RESULT 2 action log :", h.records[0])

# ------------------------------------------------------------
# Step 4 - Exception logged with context
# ------------------------------------------------------------
h.records.clear()
try:
    spark.createDataFrame([("1",), ("abc",)], ["x"]).selectExpr("cast(x as int) as n").collect()
    log.info("cast succeeded")
except Exception as e:
    log.error(f"step=cast_x status=failed error_type={type(e).__name__}")
print("RESULT 3 exception  :", h.records[0])

# ------------------------------------------------------------
# Step 5 - Structured logs: same fields in every record,
# explicit schema so inference cannot fail
# ------------------------------------------------------------
now = datetime.now(timezone.utc).isoformat()
events = [
    (now, "read_orders",  "done",   n,    None),
    (now, "cast_x",       "failed", None, "NumberFormatException"),
    (now, "write_silver", "done",   1500, None),
]
df_logs = spark.createDataFrame(
    events, "ts string, step string, status string, rows long, error_type string"
)
failed = df_logs.filter("status = 'failed'").collect()
print("RESULT 4 log rows   :", df_logs.count())
print("RESULT 5 failed step:", [(r["step"], r["error_type"]) for r in failed])

# S3.22 — Key Takeaways: Custom Logging

## 1. Why Use Logging

`print()` is useful for quick debugging, but Python logging provides a standard framework for:

- log levels
- timestamps
- logger names
- formatting
- filtering
- multiple output destinations
- exception context

A log record does **not automatically survive the run**.

Durability depends on where the logs are stored.

In our lab:

```text
ListHandler
→ Python memory
→ lost when the notebook/session ends
```

In production:

```text
Application / pipeline
        ↓
durable logging destination
        ↓
search / monitoring / retention
```

---

## 2. Python Logging Levels

Standard severity order:

```text
DEBUG
<
INFO
<
WARNING
<
ERROR
<
CRITICAL
```

Typical meaning:

```text
DEBUG
→ detailed diagnostic information

INFO
→ normal operational progress

WARNING
→ unexpected condition, but processing can continue

ERROR
→ an operation failed

CRITICAL
→ severe failure
```

The logger threshold determines which records are accepted.

Example:

```text
Logger level = INFO

DEBUG
→ filtered

INFO
WARNING
ERROR
CRITICAL
→ kept
```

With:

```text
Logger level = WARNING
```

only:

```text
WARNING
ERROR
CRITICAL
```

are kept.

---

## 3. Log Format

Useful human-readable log fields include:

```text
timestamp
level
logger name
message
```

Example:

```text
2026-11-06 ... UTC | INFO | retailpulse.s3_22 | step=count_orders status=start
```

A consistent format makes troubleshooting easier.

---

## 4. Logging Around a Spark Action

Spark transformations are lazy.

Therefore, to measure actual execution time:

```text
log start
    ↓
start timer
    ↓
Spark action
    ↓
stop timer
    ↓
log rows + duration
```

Example action:

```python
order_count = df_orders.count()
```

For elapsed duration we used:

```python
time.perf_counter()
```

The timing is meaningful because it surrounds the action that actually triggers Spark execution.

---

## 5. Exception Logging

Inside an exception handler:

```python
logger.exception(...)
```

records:

```text
log message
+
exception information
+
traceback
```

This is more useful for debugging than logging only:

```text
"something failed"
```

Useful exception context includes:

```text
step
status
error_type
```

Example:

```text
step=cast_x
status=failed
error_type=...
```

### Important

Tracebacks and exception messages can contain source values, paths or other sensitive information.

Review what is being logged before sending logs to shared or centralized destinations.

---

## 6. Observed Exception Behaviour in Our Workspace

Our controlled test attempted:

```text
"abc"
    ↓
CAST AS INT
```

With ANSI behaviour enabled, Spark produced:

```text
CAST_INVALID_INPUT
```

The Spark statement itself failed, so Databricks correctly displayed the query as failed.

However:

```text
Spark query
→ failed

Python try/except
→ caught the exception

logger.exception(...)
→ recorded the failure
```

Therefore a failed Spark statement and successful Python exception handling can both be true at the same time.

---

## 7. Structured Logging

Human-readable logging is optimized for people.

Structured logging is optimized for machines and analytics.

Example JSON event:

```json
{
  "ts": "...",
  "level": "INFO",
  "step": "count_orders",
  "status": "done",
  "rows": 1500,
  "seconds": 1.25
}
```

Useful structured fields can include:

```text
ts
level
logger
message
step
status
rows
seconds
error_type
exception
```

---

## 8. Structured Logs as Data

Structured events can be converted into a Spark DataFrame and queried using normal DataFrame operations.

Examples:

```text
filter failed events
group by status
count failures
analyze duration
identify failing steps
```

Conceptually:

```text
JSON events
    ↓
Spark DataFrame
    ↓
filter / select / groupBy
    ↓
operational analysis
```

---

## 9. Explicit Schema for Structured Logs

For structured logs, an explicit schema is safer than relying entirely on inference.

Why?

Some fields may contain only:

```text
NULL
NULL
NULL
```

for a particular batch.

Spark then has no non-null value from which to infer the column type and can raise:

```text
CANNOT_DETERMINE_TYPE
```

Therefore we defined a stable schema such as:

```text
ts          → STRING
level       → STRING
logger      → STRING
message     → STRING
step        → STRING
status      → STRING
rows        → LONG
seconds     → DOUBLE
error_type  → STRING
exception   → STRING
```

This gives the structured log data a predictable contract.

---

## 10. Human-Readable vs Structured Logs

### Human-readable

```text
2026-11-06 ... | INFO | ... | step=count_orders status=done rows=...
```

Best for:

```text
people
interactive debugging
quick investigation
```

### Structured JSON

```json
{
  "step": "count_orders",
  "status": "done",
  "rows": 1500
}
```

Best for:

```text
search
filtering
aggregation
monitoring
automation
analytics
```

---

## 11. Logging Boundary

Our Python logger runs in the notebook's Python process.

It does **not automatically capture all Spark executor/task/runtime logs**.

Conceptually:

```text
Notebook Python process
        ↓
Python logging
```

is different from:

```text
Spark distributed execution
        ↓
executor / task / runtime logging
```

They are related operationally but are not the same logging mechanism.

---

## 12. Serverless / Spark Connect

In our environment, do not use:

```python
spark.sparkContext.parallelize(...)
```

RDD/SparkContext APIs are not available in our Serverless / Spark Connect setup.

For our small structured event collection we use:

```python
spark.createDataFrame(...)
```

with an explicit schema.

---

## 13. In-Memory vs Durable Logging

### This lab

```text
ListHandler
    ↓
Python list
    ↓
easy to inspect
    ↓
lost when session ends
```

### Production

Logs normally go to a durable and searchable destination.

Depending on the architecture, this could include:

```text
Delta tables
volume/file storage
centralized logging platform
monitoring platform
```

Production logging should consider:

```text
durability
searchability
retention
monitoring
alerts
access control
security
```

---

## 14. Good Operational Log Fields

Useful fields often include:

```text
run_id
pipeline
notebook
step
status
rows
duration
source
target
error_type
```

Logs should answer:

```text
What happened?
Where?
When?
Was it successful?
How many records were processed?
How long did it take?
What failed?
```

---

## 15. Security

Do not log secrets such as:

```text
passwords
API keys
access tokens
connection secrets
```

Also avoid unnecessary sensitive payloads or personal data.

Logging should improve observability without creating a security problem.

---

## 16. Observed in Our Workspace

Verified:

```text
ANSI malformed cast test
→ CAST_INVALID_INPUT observed

Spark statement
→ failed as expected

Python try/except
→ caught the exception

logger.exception(...)
→ recorded exception context and traceback
```

Not verified yet:

```text
Records kept at INFO threshold       = ______

Records kept at WARNING threshold    = ______

TPCH order count                     = ______

Step 3 elapsed seconds               = ______

Structured log DataFrame rows        = ______

Failed structured log events         = ______

Status aggregation result            = ______
```

Fill these values from the actual notebook output before marking S3.22 complete.

---

## 17. Core Understanding

```text
Need quick debugging?
→ print() can help

Need levels, filtering and formatting?
→ Python logging

Need failure context and traceback?
→ logger.exception(...)

Need operational analytics?
→ structured JSON logs

Need Spark analysis of log events?
→ explicit schema + Spark DataFrame

Need durable production logging?
→ persistent centralized destination

Need secure logging?
→ never expose secrets or unnecessary sensitive data
```

**S3.22 — Custom Logging remains open until the remaining notebook outputs are verified.**